# Exp 03 · Architecture Deep-Dive (2): Vision Encoders (SigLIP + DINOv2)

**Goal**: turn one real camera image into the **256 visual tokens** Llama actually reads, using OpenVLA's true vision backbone — and see *why it takes two encoders*.

**No GPU needed.** One image through two ViTs is a few seconds on CPU.

## The mirror of the action side
The action notebook answered: *how do we turn a robot action into tokens Llama can emit?*
This one answers the reverse: *how do we turn an image into tokens Llama can read?*
Both do the same thing — translate non-text into the only language Llama knows: **tokens**.

```
image ──┬─→ SigLIP ─→ semantic features ┐
        └─→ DINOv2 ─→ geometric features ┘─concat→ 256 visual tokens ┐
                                                                      ├─→ Llama-2 ─→ action tokens
instruction text ───────────────────────→ text tokens ──────────────┘
```

## Why two encoders?
- **SigLIP** is trained on image+text pairs → strong at **semantics** (*what* something is: "a bowl", "red", "a cabinet"). Needed to follow `put the bowl in the cabinet`.
- **DINOv2** is trained self-supervised, no text → strong at **spatial / geometry** (*where* things are, shapes, boundaries, left-vs-right). Needed to actually reach out and grasp.
- Robot manipulation needs **both** — recognize the right object *and* localize it precisely. So OpenVLA concatenates both feature sets per patch.

OpenVLA's real backbone is **DINOSigLIP @ 224px**: DINOv2 ViT-L/14 + SigLIP SO400M/14.

---
## Step 1 · From image to patches (the arithmetic)

A ViT chops the image into a grid of fixed-size **patches**, and turns each patch into one feature vector (one "visual token").

- image is resized to **224 × 224**
- patch size = **14**
- grid = `224 / 14 = 16` per side → `16 × 16 = 256` patches

That **256** is exactly the "256 image patches" you saw in the eval logs. (Unrelated to the 256 *bins* on the action side — coincidental number.)

In [1]:
IMG_SIZE   = 224
PATCH_SIZE = 14
grid = IMG_SIZE // PATCH_SIZE
print(f'{IMG_SIZE} / {PATCH_SIZE} = {grid} patches per side')
print(f'{grid} x {grid} = {grid*grid} patches  -> {grid*grid} visual tokens per encoder')

224 / 14 = 16 patches per side
16 x 16 = 256 patches  -> 256 visual tokens per encoder


---
## Step 2 · Run the real encoders and read the shapes

We load both backbones via `timm` (the same architectures OpenVLA uses) and push one image through each.

```bash
pip install torch timm pillow
```

We reuse the first frame from Exp 01 as the input image.

In [2]:
import torch, timm
from PIL import Image
torch.set_grad_enabled(False)   # pure inference, no training

img = Image.open('../01-zero-shot-reproduction/first_frame.png').convert('RGB')
print('raw image size:', img.size)

MODELS = {
    'DINOv2': 'vit_large_patch14_reg4_dinov2.lvd142m',   # geometry / spatial
    'SigLIP': 'vit_so400m_patch14_siglip_224',           # semantics / language-aligned
}

raw image size: (256, 256)


/Users/duanxingjuan/GitPJ/openVLA-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def encode_image(name):
    # img_size=224: DINOv2 defaults to 518px, but OpenVLA forces it to 224 so both encoders
    # produce the same 16x16=256 patch grid.
    model = timm.create_model(name, pretrained=True, num_classes=0, img_size=IMG_SIZE).eval()
    cfg = timm.data.resolve_data_config({}, model=model)
    cfg['input_size'] = (3, IMG_SIZE, IMG_SIZE)          # force 224 preprocessing
    transform = timm.data.create_transform(**cfg)
    x = transform(img).unsqueeze(0)                       # [1, 3, 224, 224]
    tokens = model.forward_features(x)                    # [1, n_tokens, dim]
    patch_tokens = tokens[:, -256:, :]                    # last 256 = the patch tokens
    return x, tokens, patch_tokens

feat = {}
for tag, name in MODELS.items():
    x, tokens, patches = encode_image(name)
    feat[tag] = patches
    n_prefix = tokens.shape[1] - patches.shape[1]
    print(f'{tag:8} input {tuple(x.shape)}  all_tokens {tuple(tokens.shape)} '
          f'({n_prefix} prefix [cls/registers] + 256 patches)  patch_dim={patches.shape[-1]}')

DINOv2   input (1, 3, 224, 224)  all_tokens (1, 261, 1024) (5 prefix [cls/registers] + 256 patches)  patch_dim=1024


SigLIP   input (1, 3, 224, 224)  all_tokens (1, 256, 1152) (0 prefix [cls/registers] + 256 patches)  patch_dim=1152


### Read those shapes
- **SigLIP** → `[1, 256, 1152]`: exactly 256 patch tokens, each 1152-dim. No prefix tokens.
- **DINOv2** → `[1, 261, 1024]`: 1 `cls` + 4 `register` tokens, **then** 256 patch tokens, each 1024-dim. We slice off the 5 prefix tokens (`[:, -256:, :]`) to keep just the patch grid.
- Both end up with the **same 256 patches** — that's why we forced DINOv2 to 224px.

In [4]:
# Fuse the two encoders: concatenate their features along the channel dim, per patch.
fused = torch.cat([feat['DINOv2'], feat['SigLIP']], dim=-1)
print('DINOv2 patches :', tuple(feat['DINOv2'].shape), '(geometry)')
print('SigLIP patches :', tuple(feat['SigLIP'].shape), '(semantics)')
print('fused          :', tuple(fused.shape),
      f'-> {fused.shape[1]} visual tokens, each {fused.shape[-1]} dims (= 1024 + 1152)')
print()
print('These 256 fused vectors get linearly projected to Llama\'s hidden size and prepended')
print('to the text tokens — that is literally how the image "enters" the language model.')

DINOv2 patches : (1, 256, 1024) (geometry)
SigLIP patches : (1, 256, 1152) (semantics)
fused          : (1, 256, 2176) -> 256 visual tokens, each 2176 dims (= 1024 + 1152)

These 256 fused vectors get linearly projected to Llama's hidden size and prepended
to the text tokens — that is literally how the image "enters" the language model.


## ✅ Section summary (backfilled into LEARNING_PLAN.md Phase 2)
- [x] Vision encoder = SigLIP + DINOv2 dual encoder, features **concatenated** per patch
- [x] Saw the numbers: 224px / patch-14 → **256 patches**; DINOv2 1024-d + SigLIP 1152-d → fused **2176-d**
- [x] Understood *why two*: **SigLIP = semantics (what), DINOv2 = geometry (where)**

### Things to try
- Feed a different image and confirm you still get 256 tokens (token count depends on resolution, not content).
- Print `tokens[:, :5, :]` for DINOv2 to look at the cls + 4 register prefix tokens we sliced off.

**Next stop**: the prompt template `In: What action should the robot take to {instruction}? Out:` and how it maps to the tokenizer — then Phase 2 is complete and we move to Phase 3 (LoRA finetuning, needs a GPU).